In [ ]:
!pip install transformers datasets torch scikit-learn pandas


In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/train.csv", engine="python", on_bad_lines="skip")

print(df.shape)
print(df.columns)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(159571, 8)
Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [ ]:
toxicity_labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

df['label'] = (df[toxicity_labels].sum(axis=1) > 0).astype(int)
df = df[['comment_text', 'label']]

df['label'].value_counts()


,count
label,
0,143346
1,16225


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset


In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df['label']
)

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)


In [ ]:
print(len(train_ds), len(val_ds))
train_ds[0]


143613 15958


{'comment_text': ", 25 March 2013 (UTC)\n\nThat's some strange interpretation of bullying when I'm supposedly outnumbered with votes and all of my edits have been reverted by you. This isn't a confession is it?   22:14",
 'label': 0,
 '__index_level_0__': 81604}

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(batch):
    return tokenizer(
        batch["comment_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_ds_tokenized = train_ds.map(tokenize_function, batched=True)
val_ds_tokenized = val_ds.map(tokenize_function, batched=True)

train_ds_tokenized = train_ds_tokenized.remove_columns(["comment_text", "__index_level_0__"])
val_ds_tokenized = val_ds_tokenized.remove_columns(["comment_text", "__index_level_0__"])

train_ds_tokenized.set_format("torch")
val_ds_tokenized.set_format("torch")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/143613 [00:00<?, ? examples/s]

Map:   0%|          | 0/15958 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [ ]:
from transformers import pipeline

emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base"
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
num_labels = 2  # toxic / not toxic

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
type(model)


transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification

In [ ]:
from transformers import TrainingArguments

batch_size = 16

training_args = TrainingArguments(
    output_dir="harm_detect_results",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    # load_best_model_at_end might also not exist in very old versions
)



In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}
!ls


drive  sample_data


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="harm_detect_results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=50,
    num_train_epochs = 4,
learning_rate = 2e-5
)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

labels = train_df["label"].values
weights = compute_class_weight("balanced", classes=np.unique(labels), y=labels)
class_weights = torch.tensor(weights, dtype=torch.float)

In [ ]:
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
print(globals().keys())

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', '_', '__', '___', '_i', '_ii', '_iii', '_i1', '_exit_code', '_i2', 'pd', 'torch', 'train_test_split', 'Dataset', 'AutoTokenizer', 'AutoModelForSequenceClassification', 'TrainingArguments', 'Trainer', '_i3', 'os', '_i4', '_i5', '_i6', '_i7', '_i8', '_i9', 'drive', '_i10', 'df', '_i11', '_i12', '_i13', '_13', '_i14', '_i15', '_15', '_i16', '_16', '_i17', 'toxicity_labels', '_17', '_i18', '_i19', 'train_df', 'val_df', 'train_ds', 'val_ds', '_i20', '_20', '_i21', 'model_name', 'tokenizer', 'tokenize_function', 'train_ds_tokenized', 'val_ds_tokenized', '_i22', '_i23', 'pipeline', 'emotion_classifier', '_i24', 'num_labels', 'model', '_i25', '_25', '_i26', 'batch_size', 'training_args', '_i27', 'np', 'accuracy_score', 'f1_score', 'compute_metrics', '_i28', '_i29', '_i30', 'compute_class_weight', 'labels', 'weights', 'class_w

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_tokenized,
    eval_dataset=val_ds_tokenized
)

In [ ]:
import torch
torch.cuda.is_available()   # should print True


True

In [ ]:
trainer.train()


Step,Training Loss
50,0.598756
100,0.388628
150,0.415296
200,0.418132
250,0.218351
300,0.389656
350,0.275827
400,0.319277
450,0.306145
500,0.327189


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=35904, training_loss=0.1535986477852369, metrics={'train_runtime': 8273.2421, 'train_samples_per_second': 69.435, 'train_steps_per_second': 4.34, 'total_flos': 1.902404052318413e+16, 'train_loss': 0.1535986477852369, 'epoch': 4.0})

In [ ]:
metrics = trainer.evaluate()
metrics


{'eval_loss': 0.5820801854133606,
 'eval_runtime': 66.3241,
 'eval_samples_per_second': 240.606,
 'eval_steps_per_second': 15.047,
 'epoch': 4.0}

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_tokenized,
    eval_dataset=val_ds_tokenized,
    compute_metrics=compute_metrics
)


In [ ]:
trainer.evaluate()


{'eval_loss': 0.2246590107679367,
 'eval_model_preparation_time': 0.0013,
 'eval_accuracy': 0.9643439027447048,
 'eval_precision': 0.8213414634146341,
 'eval_recall': 0.8299445471349353,
 'eval_f1': 0.8256205945448973,
 'eval_runtime': 64.2028,
 'eval_samples_per_second': 248.556,
 'eval_steps_per_second': 15.544}

In [ ]:
import os
os.listdir("saved_models")


['harm_detect_model.pt', 'harm_detect_model_hf']

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    label = torch.argmax(probs).item()
    return label, probs.tolist()

# example
text = "I love myself"
label, confidence = predict(text)

print("Predicted label:", label)
print("Confidence:", confidence)


Predicted label: 0
Confidence: [[0.9999469518661499, 5.305017839418724e-05]]


In [ ]:
trainer.save_model("harm_detection_model")
tokenizer.save_pretrained("harm_detection_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('harm_detection_model/tokenizer_config.json',
 'harm_detection_model/tokenizer.json')

In [ ]:
!zip -r harm_detection_model.zip harm_detection_model

  adding: harm_detection_model/ (stored 0%)
  adding: harm_detection_model/config.json (deflated 49%)
  adding: harm_detection_model/tokenizer.json (deflated 71%)
  adding: harm_detection_model/tokenizer_config.json (deflated 43%)
  adding: harm_detection_model/model.safetensors (deflated 8%)
  adding: harm_detection_model/training_args.bin (deflated 53%)


In [ ]:
from google.colab import files
files.download("harm_detection_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>